# 🛡️ AI Fraud Detection Agent for SME Loans
## Interactive Demo Notebook

This notebook walks through the entire agent pipeline for detecting fraud and AML risks in SME loan applications.

## 1. Setup & Imports

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '../src')

from agents.document_ingestor import DocumentIngestor
from agents.data_extractor import DataExtractor
from agents.api_lookup_agent import APILookupAgent
from agents.anomaly_detector import AnomalyDetector
from agents.aml_checker import AMLChecker
from agents.risk_scorer import RiskScorer
from agents.report_generator import ReportGenerator

print("✅ All agents imported successfully!")

## 2. Step 1: Document Ingestion

Parse bank statement CSV file.

In [ ]:
ingestor = DocumentIngestor()
bank_data = ingestor.ingest_file('data/sample_bank_statements.csv')

print(f"Format: {bank_data['format']}")
print(f"Rows: {bank_data['row_count']}")
print(f"Columns: {bank_data['columns']}")
print(f"Sample: {bank_data['sample'][0]}")

## 3. Step 2: Data Extraction

Extract transactions and business details.

In [ ]:
extractor = DataExtractor()
extracted = extractor.extract(bank_data)

transactions = extracted['transactions']
business = extracted['business_details']

print(f"📊 Extracted {len(transactions)} transactions")
print(f"🏢 Business: {business.get('business_name', 'N/A')}")
print(f"🆔 ABN: {business.get('abn', 'N/A')}")
print("\nFirst transaction:")
print(transactions[0])

## 4. Step 3: API Lookup (Mock ABR/ATO)

Validate ABN and business registration.

In [ ]:
lookup_agent = APILookupAgent()

abn = business.get('abn', '')
name = business.get('business_name', '')

abr_result = lookup_agent.lookup_business(abn, name)
ato_result = lookup_agent.lookup_tax_return(abn)

print("=== ABR Validation ===")
print(f"Valid: {abr_result['valid']}")
print(f"Status: {abr_result['status']}")
print(f"Entity Type: {abr_result['entity_type']}")

print("\n=== ATO Tax Return ===")
print(f"Filed: {ato_result['found']}")
print(f"Reported Revenue: ${ato_result['reported_revenue']:,.2f}")

## 5. Step 4: Anomaly Detection

Use PyOD Isolation Forest + rule-based detection.

In [ ]:
detector = AnomalyDetector(contamination=0.1)
anomalies = detector.detect_anomalies(transactions)

print(f"Detected {len(anomalies)} anomalies:")
for a in anomalies:
    print(f"  • {a['type']:15s} | TX #{a['transaction_id']:3d} | ${a['amount']:>10,.2f} | {a['severity']}")

## 6. Step 5: AML Screening

Check against watchlists and high-risk patterns.

In [ ]:
aml = AMLChecker()
aml_alerts = aml.check_transactions(transactions, business)

print(f"AML Alerts: {len(aml_alerts)}")
for alert in aml_alerts:
    print(f"  🚨 {alert['type']:25s} | Severity: {alert['severity']:6s} | {alert['message']}")

## 7. Step 6: Risk Scoring

Calculate composite score + APRA fields.

In [ ]:
scorer = RiskScorer()
risk = scorer.calculate_score(anomalies, aml_alerts)

print("="*50)
print(f"📊 RISK SCORE: {risk['score']}/100")
print(f"📛 CATEGORY : {risk['category']}")
print("="*50)
print("\nPoint Breakdown:")
for key, pts in risk['breakdown'].items():
    if key != 'total_points':
        print(f"  {key}: {pts}")

print("\n--- APRA Compliance Fields ---")
apra = risk['apra_fields']
print(f"Asset Classification: {apra['asset_classification']}")
print(f"Regulatory Code:     {apra['regulatory_code']}")
print(f"Impairment Provision: ${apra['impairment_provision']:,.2f}")
print(f"LTV Ratio:            {apra['loan_to_value_ratio']:.2%}")
print(f"Manual Review:        {'Yes' if apra['requires_manual_review'] else 'No'}")

## 8. Step 7: Generate Report

Compile into PDF audit report.

In [ ]:
reporter = ReportGenerator(output_dir='reports/')
report = reporter.generate_report(
    risk_data=risk,
    anomalies=anomalies,
    aml_alerts=aml_alerts,
    business_details=business,
    task_id='demo',
    format='pdf'
)

print(f"✅ Report generated: {report['report_path']}")

## 🎉 Complete! 

You've just seen the full agent pipeline:

1. **Ingest** – ParCSV → structured data
2. **Extract** – Transactions, ABN, business name
3. **Lookup** – Validate via mock ABR/ATO
4. **Detect** – Anomalies with Isolation Forest
5. **Screen** – AML watchlist checks
6. **Score** – Risk 0-100 + APRA fields
7. **Report** – PDF audit document

### Next Steps

- Explore `src/agents/` for implementation details
- Check `docs/architecture.md` for system design
- Run `pytest tests/` to see test coverage
- Try the HTML frontend at `http://localhost:8000`

**Questions?** See `README.md` or `docs/demo_guide.md`.